# Decoding Healthcare Costs
## Analysis of NYC Hospital Trends
**DS 5010 | Sherwin Vahidimowlavi**

This notebook analyzes the 2016 NY SPARCS inpatient discharge dataset to explore and model key aspects of hospital operations across NYC boroughs:
- Patient demographics and distribution
- Length of Stay (LOS) patterns
- Cost and financial trends
- Admissions and operational insights
- Clinical outcomes and severity
- Predictive modeling for LOS, mortality risk, and total costs

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_preprocessing import (
    load_and_clean_data,
    get_info,
    filter_boroughs,
    prepare_features,
    add_borough_specific_features
)
from borough_analysis import (
    compare_operational_metrics,
    analyze_patient_demographics,
    compare_clinical_outcomes,
    calculate_borough_disparities,
    plot_pairgrid,
    analyze_top_diagnoses,
    plot_borough_comparisons
)
from modeling import (
    train_borough_cost_models,
    train_mortality_models,
    train_borough_los_models
)
from plot_utils import set_style

# Display plots inline
%matplotlib inline
set_style()

# Create figures directory if it doesn't exist
os.makedirs('figures', exist_ok=True)

---
## 1. Data Loading and Cleaning

In [ ]:
raw_df = load_and_clean_data('data/nyhospital.csv')
get_info(raw_df)

### 1.1 Filter to NYC Boroughs

ZIP code prefixes are used to assign each record to one of the five NYC boroughs. Records outside NYC are dropped.

In [ ]:
nyc_df = filter_boroughs(raw_df)
print(f"\nNYC records retained: {len(nyc_df):,}")

### 1.2 Feature Engineering

Key transformations applied:
- Numeric conversion for costs, LOS, and severity codes
- Age category mapping (Child → Senior)
- Binary `Is Emergency` and `Mortality_Label` flags
- LOS categories (Short / Medium / Long) based on quartiles
- Log-transformed costs and cost-charge ratio
- Borough-relative cost metrics

In [ ]:
processed_df = prepare_features(nyc_df)
processed_df = add_borough_specific_features(processed_df)
print(f"Final shape: {processed_df.shape}")
processed_df.head(3)

---
## 2. Patient Demographics and Distribution

### Q1: What is the patient distribution by borough?

**Observation:** Manhattan has the most patients, followed by Brooklyn and Queens.  
**Implication:** Resource allocation should prioritize high-volume boroughs.

In [ ]:
pt_count = processed_df['Borough'].value_counts()
borough_pt_pct = (pt_count / pt_count.sum()) * 100

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(
    borough_pt_pct,
    labels=borough_pt_pct.index,
    autopct='%1.1f%%',
    startangle=140
)
ax.set_title('Patient Distribution by Borough')
plt.tight_layout()
plt.savefig('figures/patient_distribution_pie.png', dpi=300)
plt.show()

### Q2–Q3: Age and Racial/Ethnic Demographics by Borough

**Key findings:**
- Brooklyn has the most evenly distributed age profile; Manhattan skews older
- Queens and Bronx have the highest proportions of "Other Race" patients, suggesting higher demand for translation services
- Brooklyn has the highest proportion of Black/African American patients

In [ ]:
analyze_patient_demographics(processed_df)

---
## 3. Length of Stay Analysis

### Q4–Q5: LOS Distribution and Categorization

LOS is bucketed into three categories based on quartiles:
- **Short:** ≤ Q1 (~3 days)
- **Medium:** Q1–Q2 (~3–7 days)
- **Long:** > Q2 (7+ days)

**Observation:** Most stays are short-term, with a right-skewed distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
sns.histplot(processed_df['Length of Stay'], bins=30, kde=True, ax=axes[0])
axes[0].set_xlim(0, 30)
axes[0].set_title('Distribution of Length of Stay')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Patient Count')

# Category counts
sns.countplot(
    data=processed_df,
    x='LOS Category',
    order=['Short', 'Medium', 'Long'],
    ax=axes[1]
)
axes[1].set_title('Patient Count by LOS Category')

plt.tight_layout()
plt.savefig('figures/los_distribution.png', dpi=300)
plt.show()

### Q6: How does LOS vary by age group?

**Observation:** Older adults and seniors dominate long-stay patients. Children are the primary short-stay group, largely driven by childbirth.

In [ ]:
sns.histplot(
    data=processed_df,
    x='LOS Category',
    hue='Age Category',
    hue_order=['Child', 'Young Adult', 'Adult', 'Older Adult', 'Senior'],
    multiple='stack',
    discrete=True,
    stat='count'
)
plt.title('Length of Stay by Age')
plt.xticks(rotation=45)
plt.savefig('figures/los_by_age.png', dpi=300)
plt.show()

### Q7–Q8: LOS vs. Cost and Average LOS by Borough

**Observations:**
- Costs plateau around 21 days; Staten Island has the highest cost per day beyond that point
- Manhattan has the longest average LOS; Staten Island the shortest

In [ ]:
compare_operational_metrics(processed_df)

### Q9: LOS vs. Illness Severity

**Observation:** Severity and LOS have a strong positive relationship. Extreme-severity cases are responsible for the longest stays.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=processed_df,
    x='APR Severity of Illness Description',
    y='Length of Stay',
    order=['Minor', 'Moderate', 'Major', 'Extreme'],
    showfliers=False
)
plt.title('Length of Stay by Illness Severity')
plt.tight_layout()
plt.savefig('figures/los_by_severity.png', dpi=300)
plt.show()

---
## 4. Costs and Financial Trends

### Q10: Key Numerical Trends (Pair Grid)

**Observations:**
- Illness severity has the strongest correlation with total cost
- APR DRG code correlates with cost but not strongly with LOS
- Manhattan has high short-stay costs

In [ ]:
plot_pairgrid(processed_df)

### Q11–Q12: Diagnoses with Highest Costs and Daily Cost Efficiency

**Observations:**
- Sepsis is the most expensive diagnosis; childbirth is the most common but least costly
- Brooklyn is the most cost-efficient borough; Staten Island the least

In [ ]:
analyze_top_diagnoses(processed_df, top_n=10)

### Q13–Q14: Drug Code vs. Diagnosis and Cost

**Observations:**
- Lower APR DRG codes (acute care, e.g. organ transplants) are highest cost
- The most commonly administered drug groups are not the most expensive ones

*(Visualizations for Q13–Q14 are generated as part of `compare_operational_metrics` above — see `figures/drg_vs_diagnosis_heatmap.png` and `figures/apr_drg_vs_costs.png`)*

---
## 5. Admissions and Operational Insights

### Q16: Payment Types by Borough

**Observations:**
- Medicare/Medicaid dominate across all boroughs; the Bronx has the highest proportion
- Manhattan has the highest private insurance share, consistent with its younger patient profile

*(Payment mix visualization generated within `analyze_patient_demographics` above — see `figures/payment_by_borough.png`)*

### Q17: Emergency Admission Rates by Borough

**Observations:**
- Staten Island (67.8%) and Bronx (66.3%) have the highest emergency admission rates
- Brooklyn is the most cost-efficient despite similar emergency rates to other boroughs

*(Lollipop chart generated within `compare_operational_metrics` above — see `figures/emergency_rates_lollipop.png`)*

### Borough Comparison Summary

In [ ]:
metrics = calculate_borough_disparities(processed_df)
print(metrics.round(2))
plot_borough_comparisons(metrics)

---
## 6. Clinical Outcomes and Severity

### Q18–Q20: Diagnoses, Severity Distribution, and Borough Breakdown

**Observations:**
- Childbirth, psychiatric, metabolic syndrome, and substance abuse are the four dominant diagnosis categories
- Severity distribution is nearly identical across boroughs
- Extreme cases are a minority of volume but the majority of cost

In [ ]:
compare_clinical_outcomes(processed_df)

### Q21: Mortality Rates by Borough

**Observations:**
- Staten Island has the highest extreme-mortality rate; Bronx the lowest
- Warrants investigation into care quality disparities

*(Mortality dot plot generated within `compare_clinical_outcomes` above — see `figures/mortality_dotplot.png`)*

### Q23: Clinical Outcomes by Race

**Observation:** Extreme mortality risk is highest among Black/African American patients relative to their total volume, suggesting barriers to equitable care that warrant further investigation.

*(Mortality by race chart generated within `analyze_patient_demographics` above — see `figures/mortality_by_race.png`)*

---
## 7. Predictive Modeling

Three Random Forest models are trained, each evaluated per borough where applicable:

| Model | Type | Target |
|---|---|---|
| Cost model | Regressor | Length of Stay (by borough) |
| Mortality model | Classifier | Extreme mortality risk |
| LOS model | Regressor | Length of Stay (by borough) |

**Features used:**
- APR Severity of Illness Code
- APR DRG Code
- APR MDC Code
- Age Group Code
- Is Emergency
- Total Costs *(excluded for cost/LOS models to avoid leakage)*

### Q15: Which features predict total cost?

**Observation:** APR DRG Code and illness severity are the top predictors of cost across all boroughs.

In [ ]:
cost_models, cost_importance_df = train_borough_cost_models(processed_df)
print("Feature importance by borough (cost model):")
cost_importance_df

### Q22: Which features impact mortality risk?

**Observation:** Illness severity and total costs are the dominant predictors. Emergency admission status has the lowest importance.

In [ ]:
mortality_model, mortality_importance = train_mortality_models(processed_df)
print("Feature importance (mortality model):")
print(mortality_importance.sort_values(ascending=False))

### Q25: Which features predict LOS?

**Observation:** Total Costs dominates LOS prediction (~0.68–0.74 importance), followed by APR MDC and DRG codes.

In [ ]:
los_models, los_importance_df = train_borough_los_models(processed_df)
print("Feature importance by borough (LOS model):")
los_importance_df

---
## 8. Conclusions

### Cost Drivers
- **Illness severity** is the single strongest predictor of cost across all boroughs
- **APR DRG codes 001–100** (acute care / organ transplants) are disproportionately expensive
- Sepsis is the most costly diagnosis; childbirth is high-volume but low-cost

### Operational Opportunities
- **Brooklyn** is the most cost-efficient borough and can serve as an operational benchmark
- Standardizing drug formularies could reduce unnecessary high-cost prescriptions
- Staten Island and Bronx need expanded ER capacity (67.8% and 66.3% emergency admission rates)

### Cost Planning
- Use total costs + severity for capacity and budget forecasting
- Prioritize elder care infrastructure in Manhattan (highest elderly patient share)
- Flag high-severity patients on admission for early intervention to reduce LOS and cost

### Limitations and Future Work
- Dataset is a single year (2016) — no temporal trends available
- No pharmacy-level cost breakdown to tie drug codes directly to spend
- Models predict feature importance but are not deployed for inference — next step is a real-time risk-flagging pipeline